In [2]:
# 1️⃣ Imports
from langchain_community.llms import Ollama
from langchain_community.embeddings import OllamaEmbeddings
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser


# 2️⃣ Load LLM (smollm2:360m)
llm = Ollama(model="smollm2:360m")

# 3️⃣ Load Embeddings (same model)
embeddings = OllamaEmbeddings(model="smollm2:360m")

# 4️⃣ Load Document
loader = TextLoader("sample.txt")  # put your file name here
documents = loader.load()

# 5️⃣ Split into Chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)
chunks = text_splitter.split_documents(documents)

# 6️⃣ Create Vector Store
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="./chroma_db"
)

retriever = vectorstore.as_retriever()

# 7️⃣ Prompt Template
template = """
You are an intelligent assistant.
Answer strictly based on the provided context.

Context:
{context}

Question:
{question}
"""

prompt = ChatPromptTemplate.from_template(template)
parser = StrOutputParser()


# 8️⃣ RAG Function
def ask_question(question):
    docs = retriever.get_relevant_documents(question)
    context = "\n".join([doc.page_content for doc in docs])

    formatted_prompt = prompt.format(
        context=context,
        question=question
    )

    response = llm.invoke(formatted_prompt)
    return parser.parse(response)


# 9️⃣ Ask
query = "What is this document about?"
answer = ask_question(query)

print("\nAnswer:\n")
print(answer)

C:\Users\Monalika\AppData\Local\Temp\ipykernel_21696\1843652513.py:13: LangChainDeprecationWarning: The class `Ollama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaLLM``.
  llm = Ollama(model="smollm2:360m")
C:\Users\Monalika\AppData\Local\Temp\ipykernel_21696\1843652513.py:16: LangChainDeprecationWarning: The class `OllamaEmbeddings` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaEmbeddings``.
  embeddings = OllamaEmbeddings(model="smollm2:360m")


RuntimeError: Error loading sample.txt